In [1]:
import os, gc, glob, warnings
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import pyarrow as pa
from collections import defaultdict
warnings.filterwarnings('ignore')

BD_FINAL        = 7
K1_FINAL        = 1.0
K2_FINAL        = 2.0
K3_FINAL        = 3.5
MIN_PERSISTENCE = 4
LABEL_LOGIC     = 'OR'

GUARDRAILS = {
    'network.latency': {'media': 0.250, 'alta': 0.350},
    'network.loss':    {'media': 3.0,   'alta': 8.0},
    'network.jitter':  {'media': 7.0,   'alta': 10.0},
}
MIN_MAD = {
    'network.latency': 0.005,
    'network.loss':    0.5,
    'network.jitter':  0.1,
}

START_DATE = pd.Timestamp('2026-04-23', tz='UTC')
BD_END     = START_DATE + pd.Timedelta(days=BD_FINAL)

SPLIT_DATES = {
    'train': (pd.Timestamp('2026-04-30', tz='UTC'), pd.Timestamp('2026-08-16', tz='UTC')),
    'val':   (pd.Timestamp('2026-08-16', tz='UTC'), pd.Timestamp('2026-09-08', tz='UTC')),
    'test':  (pd.Timestamp('2026-09-08', tz='UTC'), pd.Timestamp('2026-10-01', tz='UTC')),
}

WINDOW_MIN  = 30
SAMPLES_W   = 6
METRICS_NET = ['network.latency', 'network.loss', 'network.jitter']

SAVE_DIR  = '/srv/work'

ARCHIVOS = sorted(
    glob.glob(f'{SAVE_DIR}/te_*_raw_may_w*.parquet') +
    glob.glob(f'{SAVE_DIR}/te_*_raw_w[0-9].parquet') +
    glob.glob(f'{SAVE_DIR}/te_*_raw_apr_w*.parquet') +
    glob.glob(f'{SAVE_DIR}/te_*_raw_jul_w*.parquet') +
    glob.glob(f'{SAVE_DIR}/te_*_raw_aug_w*.parquet') +
    glob.glob(f'{SAVE_DIR}/te_*_raw_sep_w*.parquet')
)

OUT_LABEL = f'{SAVE_DIR}/te_labeled_ventanas_v4.parquet'
OUT_BASE  = f'{SAVE_DIR}/te_baseline_stats_v4.parquet'

print(f"Archivos   : {len(ARCHIVOS)}  (esperado: 72)")
if len(ARCHIVOS) != 72:
    print(f"  WARNING: se esperaban 72, encontrados {len(ARCHIVOS)}")
print(f"Baseline   : {START_DATE.date()} -> {BD_END.date()} ({BD_FINAL}d — Apr22 EXCLUIDO)")
print(f"Eval desde : {BD_END.date()}")
print(f"Salida     : {OUT_LABEL}")
print(f"\nPARAMETROS CONGELADOS v7: BD={BD_FINAL}d  K=({K1_FINAL},{K2_FINAL},{K3_FINAL})  pers>={MIN_PERSISTENCE}  logic={LABEL_LOGIC}")
print(f"\nSPLIT_DATES (Opción B 70/15/15 — LOG-37):")
for split, (s, e) in SPLIT_DATES.items():
    dias = (e - s).days
    print(f"  {split:5s}: {s.date()} -> {(e - pd.Timedelta(days=1)).date()}  ({dias}d)")


Archivos   : 72  (esperado: 72)
Baseline   : 2026-04-23 -> 2026-04-30 (7d — Apr22 EXCLUIDO)
Eval desde : 2026-04-30
Salida     : /srv/work/te_labeled_ventanas_v4.parquet

PARAMETROS CONGELADOS v7: BD=7d  K=(1.0,2.0,3.5)  pers>=4  logic=OR

SPLIT_DATES (Opción B 70/15/15 — LOG-37):
  train: 2026-04-30 -> 2026-08-15  (108d)
  val  : 2026-08-16 -> 2026-09-07  (23d)
  test : 2026-09-08 -> 2026-09-30  (23d)


In [2]:
PROVIDER_PREFIXES = ['aws', 'azure', 'gcp']
route_keys = set()

for prov in PROVIDER_PREFIXES:
    prov_files = [f for f in ARCHIVOS if f'/te_{prov}_' in f.lower()]
    if not prov_files:
        print(f"WARNING: Sin archivos para {prov}")
        continue
    for f in prov_files:
        tbl = pq.read_table(f, columns=['source_agent','cloud_provider','region'])
        df_r = tbl.to_pandas().drop_duplicates()
        del tbl; gc.collect()
        for row in df_r.itertuples():
            if pd.notna(row.source_agent):
                route_keys.add((row.source_agent, row.cloud_provider, row.region))
        del df_r; gc.collect()
    prov_count = sum(1 for r in route_keys if r[1].lower() == prov)
    print(f"  {prov.upper()}: {prov_count} rutas")

routes_df = pd.DataFrame(list(route_keys), columns=['source_agent','cloud_provider','region'])
routes_df = routes_df.sort_values(['cloud_provider','region','source_agent']).reset_index(drop=True)
print(f"\nRutas totales: {len(routes_df):,}  (esperado: ~621)")
print(routes_df.groupby('cloud_provider').size())


  AWS: 405 rutas
  AZURE: 315 rutas
  GCP: 315 rutas

Rutas totales: 1,035  (esperado: ~621)
cloud_provider
AWS      405
Azure    315
GCP      315
dtype: int64


In [3]:
PROVIDER_MAP = {
    'aws':   [f for f in ARCHIVOS if '/te_aws_'   in f.lower()],
    'azure': [f for f in ARCHIVOS if '/te_azure_' in f.lower()],
    'gcp':   [f for f in ARCHIVOS if '/te_gcp_'   in f.lower()],
}

baseline_records = []

for prov, files in PROVIDER_MAP.items():
    print(f"[{prov.upper()}] {len(files)} archivos...")
    prov_chunks = []

    for f in files:
        tbl = pq.read_table(f, columns=['_time','source_agent','cloud_provider','region','metric','value'])
        df  = tbl.to_pandas(); del tbl; gc.collect()
        df['_time'] = pd.to_datetime(df['_time'], utc=True)
        mask = (
            (df['_time'] >= START_DATE) & (df['_time'] < BD_END) &
            (df['metric'].isin(METRICS_NET)) &
            df['source_agent'].notna()
        )
        chunk = df[mask].copy(); del df; gc.collect()
        if len(chunk) > 0:
            prov_chunks.append(chunk)

    if not prov_chunks:
        print(f"  WARNING: Sin datos baseline para {prov}")
        continue

    prov_df = pd.concat(prov_chunks, ignore_index=True)
    del prov_chunks; gc.collect()

    for metric in METRICS_NET:
        mdf = prov_df[prov_df['metric'] == metric]
        grp = mdf.groupby(['source_agent','cloud_provider','region'])['value']
        p95 = grp.quantile(0.95).rename('p95')
        mad = grp.apply(lambda x: np.median(np.abs(x.values - np.median(x.values)))).rename('mad')
        stats = pd.concat([p95, mad], axis=1).reset_index()
        stats['metric'] = metric
        baseline_records.append(stats)
        del mdf; gc.collect()

    del prov_df; gc.collect()
    print(f"  OK {prov.upper()} listo")

baseline_df = pd.concat(baseline_records, ignore_index=True)
baseline_df['mad'] = baseline_df.apply(lambda r: max(r['mad'], MIN_MAD[r['metric']]), axis=1)

baseline_df.to_parquet(OUT_BASE, index=False)
print(f"\nBaseline guardado: {OUT_BASE}")
print(f"Shape: {baseline_df.shape}  (esperado ~1,863 = 621 rutas x 3 metricas)")
print(baseline_df.groupby('metric')[['p95','mad']].describe().round(4))


[AWS] 24 archivos...
  OK AWS listo
[AZURE] 24 archivos...
  OK AZURE listo
[GCP] 24 archivos...
  OK GCP listo

Baseline guardado: /srv/work/te_baseline_stats_v4.parquet
Shape: (3105, 6)  (esperado ~1,863 = 621 rutas x 3 metricas)
                    p95                                                  \
                  count    mean      std    min     25%     50%     75%   
metric                                                                    
network.jitter   1035.0  2.3546   2.8600  0.383  0.5495  0.8765  2.7956   
network.latency  1035.0  0.1588   0.0843  0.001  0.0995  0.1539  0.2022   
network.loss     1035.0  5.1706  10.6976  0.000  0.0000  0.0200  6.0000   

                              mad                                         \
                      max   count    mean     std    min     25%     50%   
metric                                                                     
network.jitter    17.5919  1035.0  0.4214  0.6545  0.100  0.1224  0.1633   
network.laten

In [4]:
baseline_df = pd.read_parquet(OUT_BASE)
baseline = {}
for row in baseline_df.itertuples():
    key = (row.source_agent, row.cloud_provider, row.region, row.metric)
    baseline[key] = {'p95': row.p95, 'mad': row.mad}

print(f"Baseline cargado: {len(baseline):,} entradas  (esperado ~1,863)")

PROVIDER_MAP = {
    'aws':   [f for f in ARCHIVOS if '/te_aws_'   in f.lower()],
    'azure': [f for f in ARCHIVOS if '/te_azure_' in f.lower()],
    'gcp':   [f for f in ARCHIVOS if '/te_gcp_'   in f.lower()],
}

labeled_chunks = []

for prov, files in PROVIDER_MAP.items():
    print(f"\n[{prov.upper()}] {len(files)} archivos...")
    for fi, f in enumerate(files):
        tbl = pq.read_table(f, columns=['_time','source_agent','cloud_provider','region','metric','value'])
        df  = tbl.to_pandas(); del tbl; gc.collect()
        df['_time'] = pd.to_datetime(df['_time'], utc=True)
        df = df[
            (df['_time'] >= BD_END) &
            (df['metric'].isin(METRICS_NET)) &
            df['source_agent'].notna()
        ].copy()
        if df.empty:
            del df; gc.collect()
            continue

        df['window'] = df['_time'].dt.floor('30min')

        file_windows = []
        for (sa, cp, reg, metric, win), grp in df.groupby(
                ['source_agent','cloud_provider','region','metric','window'], sort=False):
            bkey = (sa, cp, reg, metric)
            if bkey not in baseline:
                continue

            p95 = baseline[bkey]['p95']
            mad = baseline[bkey]['mad']
            t_baja  = p95 + K1_FINAL * mad
            t_media = max(p95 + K2_FINAL * mad, GUARDRAILS[metric]['media'])
            t_alta  = max(p95 + K3_FINAL * mad, GUARDRAILS[metric]['alta'])

            vals = grp['value'].values
            if len(vals) == 0:
                continue

            if   np.sum(vals > t_alta)  >= MIN_PERSISTENCE: lbl = 3
            elif np.sum(vals > t_media) >= MIN_PERSISTENCE: lbl = 2
            elif np.sum(vals > t_baja)  >= MIN_PERSISTENCE: lbl = 1
            else:                                            lbl = 0

            file_windows.append({
                'source_agent': sa, 'cloud_provider': cp, 'region': reg,
                'window': win, 'metric': metric, 'label': lbl
            })

        del df; gc.collect()

        if not file_windows:
            continue

        file_df = pd.DataFrame(file_windows)
        del file_windows; gc.collect()

        file_labeled = (
            file_df
            .groupby(['source_agent','cloud_provider','region','window'])['label']
            .max()
            .reset_index()
            .rename(columns={'label': 'label_ventana'})
        )
        labeled_chunks.append(file_labeled)
        del file_df, file_labeled; gc.collect()

        n_acum = sum(len(c) for c in labeled_chunks)
        print(f"  [{fi+1:02d}/{len(files)}] {os.path.basename(f)} — chunks={len(labeled_chunks)}  filas_acum={n_acum:,}")

    print(f"  OK {prov.upper()} listo")

labeled_df = pd.concat(labeled_chunks, ignore_index=True)
del labeled_chunks; gc.collect()

labeled_df = (
    labeled_df
    .groupby(['source_agent','cloud_provider','region','window'])['label_ventana']
    .max()
    .reset_index()
)

print(f"\nVentanas totales: {len(labeled_df):,}")
dist = labeled_df['label_ventana'].value_counts(normalize=True).sort_index() * 100
labels_map = {0:'Normal', 1:'Baja', 2:'Media', 3:'Alta'}
for k, pct in dist.items():
    print(f"  {labels_map[k]:8s} ({k}): {pct:.1f}%")
print(f"\n  No-normal total: {dist[dist.index > 0].sum():.1f}%")


Baseline cargado: 3,105 entradas  (esperado ~1,863)

[AWS] 24 archivos...
  [01/24] te_aws_raw_apr_w1.parquet — chunks=1  filas_acum=19,440
  [02/24] te_aws_raw_aug_w1.parquet — chunks=2  filas_acum=155,520
  [03/24] te_aws_raw_aug_w2.parquet — chunks=3  filas_acum=291,348
  [04/24] te_aws_raw_aug_w3.parquet — chunks=4  filas_acum=426,950
  [05/24] te_aws_raw_aug_w4.parquet — chunks=5  filas_acum=562,924
  [06/24] te_aws_raw_aug_w5.parquet — chunks=6  filas_acum=698,987
  [07/24] te_aws_raw_jul_w1.parquet — chunks=7  filas_acum=834,522
  [08/24] te_aws_raw_may_w1.parquet — chunks=8  filas_acum=970,278
  [09/24] te_aws_raw_may_w2.parquet — chunks=9  filas_acum=1,105,871
  [10/24] te_aws_raw_may_w3.parquet — chunks=10  filas_acum=1,241,852
  [11/24] te_aws_raw_may_w4.parquet — chunks=11  filas_acum=1,375,460
  [12/24] te_aws_raw_may_w5.parquet — chunks=12  filas_acum=1,453,220
  [13/24] te_aws_raw_sep_w1.parquet — chunks=13  filas_acum=1,607,002
  [14/24] te_aws_raw_sep_w2.parquet — chun

In [5]:

def assign_split(ts):
    train_start, train_end = SPLIT_DATES['train']
    val_start,   val_end   = SPLIT_DATES['val']
    test_start,  test_end  = SPLIT_DATES['test']
    if   train_start <= ts < train_end: return 'train'
    elif val_start   <= ts < val_end:   return 'val'
    elif test_start  <= ts < test_end:  return 'test'
    else:                               return 'other'

labeled_df['split'] = labeled_df['window'].apply(assign_split)

print("=== Distribucion por split (Opción B 70/15/15 — LOG-37) ===")
for split in ['train', 'val', 'test', 'other']:
    sdf = labeled_df[labeled_df['split'] == split]
    if sdf.empty:
        print(f"\n{split.upper()}: sin datos")
        continue
    n = len(sdf)
    dist = sdf['label_ventana'].value_counts(normalize=True).sort_index() * 100
    labels_map = {0:'Normal', 1:'Baja', 2:'Media', 3:'Alta'}
    w_min, w_max = sdf['window'].min(), sdf['window'].max()
    print(f"\n{split.upper()} ({n:,} ventanas | {str(w_min)[:10]} -> {str(w_max)[:10]}):")
    for k in [0,1,2,3]:
        pct = dist.get(k, 0)
        bar = 'X' * int(pct / 3)
        print(f"  {labels_map[k]:8s} ({k}): {pct:.1f}%  {bar}")
    no_norm = dist[dist.index > 0].sum()
    print(f"  No-normal total: {no_norm:.1f}%")

labeled_df.to_parquet(OUT_LABEL, index=False)
print(f"\nOK Guardado: {OUT_LABEL}")
print(f"   Shape: {labeled_df.shape}")
print(f"   Columnas: {list(labeled_df.columns)}")
print(f"\n   Splits:")
print(labeled_df['split'].value_counts())


=== Distribucion por split (Opción B 70/15/15 — LOG-37) ===

TRAIN (5,349,594 ventanas | 2026-04-30 -> 2026-08-15):
  Normal   (0): 86.4%  XXXXXXXXXXXXXXXXXXXXXXXXXXXX
  Baja     (1): 9.5%  XXX
  Media    (2): 2.1%  
  Alta     (3): 2.0%  
  No-normal total: 13.6%

VAL (1,140,847 ventanas | 2026-08-16 -> 2026-09-07):
  Normal   (0): 89.9%  XXXXXXXXXXXXXXXXXXXXXXXXXXXXX
  Baja     (1): 6.2%  XX
  Media    (2): 2.0%  
  Alta     (3): 1.9%  
  No-normal total: 10.1%

TEST (1,138,360 ventanas | 2026-09-08 -> 2026-09-30):
  Normal   (0): 82.4%  XXXXXXXXXXXXXXXXXXXXXXXXXXX
  Baja     (1): 12.5%  XXXX
  Media    (2): 3.0%  X
  Alta     (3): 2.1%  
  No-normal total: 17.6%

OTHER: sin datos

OK Guardado: /srv/work/te_labeled_ventanas_v4.parquet
   Shape: (7628801, 6)
   Columnas: ['source_agent', 'cloud_provider', 'region', 'window', 'label_ventana', 'split']

   Splits:
split
train    5349594
val      1140847
test     1138360
Name: count, dtype: int64


In [6]:
IN_LABEL = f'{SAVE_DIR}/te_labeled_ventanas_v4.parquet'

if not os.path.exists(IN_LABEL):
    print('SKIP — te_labeled_ventanas_v4.parquet no encontrado. Ejecutar celda 4 primero.')
else:
    labeled_df = pd.read_parquet(IN_LABEL)
    labeled_df['window'] = pd.to_datetime(labeled_df['window'], utc=True)
    labels_map = {0: 'Normal', 1: 'Baja', 2: 'Media', 3: 'Alta'}

    print(f'Archivo : {IN_LABEL}')
    print(f'Shape   : {labeled_df.shape}')
    print(f'Columnas: {list(labeled_df.columns)}')

    print('\n=== Ventanas por split ===')
    print(labeled_df['split'].value_counts())

    print('\n=== Distribucion de clases por split (Opción B 70/15/15 — LOG-37) ===')
    for split in ['train', 'val', 'test']:
        sdf = labeled_df[labeled_df['split'] == split]
        if sdf.empty:
            print(f'\n{split.upper()}: sin datos')
            continue
        n    = len(sdf)
        dist = sdf['label_ventana'].value_counts(normalize=True).sort_index() * 100
        w_min, w_max = sdf['window'].min(), sdf['window'].max()
        print(f'\n{split.upper()} ({n:,} ventanas | {str(w_min)[:10]} -> {str(w_max)[:10]}):')
        for k in [0, 1, 2, 3]:
            pct = dist.get(k, 0.0)
            bar = 'X' * int(pct / 3)
            print(f'  {labels_map[k]:8s} ({k}): {pct:5.1f}%  {bar}')
        no_norm = dist[dist.index > 0].sum()
        print(f'  No-normal total: {no_norm:.1f}%')

    print('\n=== Verificación: % No-normal por split (debe ser similar) ===')
    for split in ['train', 'val', 'test']:
        sdf = labeled_df[labeled_df['split'] == split]
        if sdf.empty:
            continue
        dist = sdf['label_ventana'].value_counts(normalize=True).sort_index() * 100
        no_norm = dist[dist.index > 0].sum()
        print(f'  {split:5s}: {no_norm:.1f}% no-normal')
    print('  (diferencia grande indicaría distribución de clases muy heterogénea)')


Archivo : /srv/work/te_labeled_ventanas_v4.parquet
Shape   : (7628801, 6)
Columnas: ['source_agent', 'cloud_provider', 'region', 'window', 'label_ventana', 'split']

=== Ventanas por split ===
split
train    5349594
val      1140847
test     1138360
Name: count, dtype: int64

=== Distribucion de clases por split (Opción B 70/15/15 — LOG-37) ===

TRAIN (5,349,594 ventanas | 2026-04-30 -> 2026-08-15):
  Normal   (0):  86.4%  XXXXXXXXXXXXXXXXXXXXXXXXXXXX
  Baja     (1):   9.5%  XXX
  Media    (2):   2.1%  
  Alta     (3):   2.0%  
  No-normal total: 13.6%

VAL (1,140,847 ventanas | 2026-08-16 -> 2026-09-07):
  Normal   (0):  89.9%  XXXXXXXXXXXXXXXXXXXXXXXXXXXXX
  Baja     (1):   6.2%  XX
  Media    (2):   2.0%  
  Alta     (3):   1.9%  
  No-normal total: 10.1%

TEST (1,138,360 ventanas | 2026-09-08 -> 2026-09-30):
  Normal   (0):  82.4%  XXXXXXXXXXXXXXXXXXXXXXXXXXX
  Baja     (1):  12.5%  XXXX
  Media    (2):   3.0%  X
  Alta     (3):   2.1%  
  No-normal total: 17.6%

=== Verificación: 

In [7]:
IN_LABEL = f'{SAVE_DIR}/te_labeled_ventanas_v4.parquet'

if not os.path.exists(IN_LABEL):
    print('SKIP — te_labeled_ventanas_v4.parquet no encontrado. Ejecutar celda 4 primero.')
else:
    labeled_df = pd.read_parquet(IN_LABEL)
    labeled_df['window'] = pd.to_datetime(labeled_df['window'], utc=True)
    test_df = labeled_df[labeled_df['split'] == 'test'].copy()

    if test_df.empty:
        print('TEST sin datos en el archivo etiquetado.')
    else:
        test_df = test_df.sort_values('window').reset_index(drop=True)
        t_min, t_max = test_df['window'].min(), test_df['window'].max()
        expected = pd.date_range(t_min, t_max, freq='30min')
        n_routes = test_df[['source_agent','cloud_provider','region']].drop_duplicates().shape[0]

        print('=== Cobertura del test set (Sep 8 – Sep 30, 23d — Opción B) ===')
        print(f'  Periodo     : {str(t_min)[:19]} -> {str(t_max)[:19]}')
        print(f'  Dias        : {(t_max - t_min).days + 1}  (esperado: 23)')
        print(f'  Rutas       : {n_routes:,}')
        print(f'  Ventanas    : {len(test_df):,}')
        total_esp = len(expected) * n_routes
        print(f'  Esperadas   : {total_esp:,}  ({len(expected)} slots x {n_routes} rutas)')
        print(f'  Cobertura   : {100 * len(test_df) / total_esp:.1f}%')

        labels_map = {0: 'Normal', 1: 'Baja', 2: 'Media', 3: 'Alta'}
        dist = test_df['label_ventana'].value_counts(normalize=True).sort_index() * 100
        print('\n  Distribucion de clases en test:')
        for k in [0, 1, 2, 3]:
            pct = dist.get(k, 0.0)
            bar = 'X' * int(pct / 3)
            print(f'    {labels_map[k]:8s} ({k}): {pct:5.1f}%  {bar}')


=== Cobertura del test set (Sep 8 – Sep 30, 23d — Opción B) ===
  Periodo     : 2026-09-08 00:00:00 -> 2026-09-30 23:30:00
  Dias        : 23  (esperado: 23)
  Rutas       : 1,035
  Ventanas    : 1,138,360
  Esperadas   : 1,142,640  (1104 slots x 1035 rutas)
  Cobertura   : 99.6%

  Distribucion de clases en test:
    Normal   (0):  82.4%  XXXXXXXXXXXXXXXXXXXXXXXXXXX
    Baja     (1):  12.5%  XXXX
    Media    (2):   3.0%  X
    Alta     (3):   2.1%  
